# 判断是否需要继续检索

检索不是越多越好。已有资料足以回答时，不应重复访问资料库；简单问题一次已经找全回答所需内容时，也不应为了“更深入”固定再查一次。这类根据问题和已找到资料决定检索次数的做法，常称为 Adaptive Retrieval（自适应检索）。

下面使用问题集中的问题和《南瓜书》页面。需要权限过滤的情形放在单独的 [检索前过滤无权访问的资料](检索前过滤无权访问的资料.ipynb) 中，不在这里重复。

In [1]:
import re
import sys
from pathlib import Path


def find_course_root(start):
    for folder in (start, *start.parents):
        if (folder / "data" / "dataset/manifest.json").is_file():
            return folder
    raise FileNotFoundError("没有找到教程数据目录，请从本节所在目录运行。")


course_root = find_course_root(Path.cwd())
if str(course_root) not in sys.path:
    sys.path.insert(0, str(course_root))

In [2]:
from common.eval_utils import emit_tutorial_audit

import json
from common.eval_utils import build_bm25_chunk_search, load_query_catalog, load_default_chunks
from common.nontraining_utils import load_annotation, load_query_controls

cases = {item["id"]: item for item in load_query_catalog()}
search = build_bm25_chunk_search(load_default_chunks())


def evidence_coverage(results, expected_pages):
    expected = set(expected_pages)
    found = {page for item in results for page in item.pages}
    return len(expected.intersection(found)) / len(expected)

def standard_metrics(results, expected_pages):
    pages = [int(page) for item in results for page in item.pages]
    expected = set(int(page) for page in expected_pages)
    found = {page for page in pages if page in expected}
    first = next((rank for rank, page in enumerate(pages, 1) if page in expected), None)
    return {'pages': pages, 'first_required_rank': first, 'required_page_coverage': len(found) / len(expected) if expected else 0.0}

def emit_standard(method, role, case_id, before, after, expected_pages, check_purpose=None, comparison=None):
    payload = {'case_id': case_id, 'method': method, 'role': role,
               'before': standard_metrics(before, expected_pages),
               'after': standard_metrics(after, expected_pages)}
    if check_purpose:
        payload['check_purpose'] = check_purpose
    if comparison is not None:
        payload['comparison'] = comparison
    emit_tutorial_audit(payload)


## 已核验的上下文足以回答，就不要再查

用户先问连续型标记对应什么任务，随后根据刚刚核验过的同一段资料追问离散型标记。基础方案每轮都检索一次；改动后的规则先检查现有资料是否已经包含回答所需的内容。这里的改进是省掉重复访问，答案本身要保持不变。

In [3]:
case = cases["continuous_label_task"]
steps = load_query_controls(case["id"])["steps"]
previous_question = steps[0]["query"]
question = steps[1]["query"]

def counted_search(counter, query, top_k=1):
    counter["calls"] += 1
    return search(query, top_k=top_k)


previous_counter = {"calls": 0}
previous_results = counted_search(previous_counter, previous_question, top_k=1)
retrieved_context = "".join(item.text for item in previous_results)
retrieved_pages = [page for item in previous_results for page in item.pages]

def context_has_topic(text, topic):
    return bool(topic) and topic in "".join(text.split())

followup_topic = next((term for term in ("离散型", "连续型") if term in question), "")
context_is_enough = context_has_topic(retrieved_context, followup_topic)
baseline_counter = {"calls": 0}
baseline_results = counted_search(baseline_counter, question)
after_check_counter = {"calls": 0}
after_check_results = [] if context_is_enough else counted_search(after_check_counter, question)
required_points = (("离散型", "分类"),)

def point_coverage(text):
    compact = "".join(text.split())
    checks = [all(part in compact for part in point) for point in required_points]
    return sum(checks) / len(checks), checks

baseline_points, baseline_checks = point_coverage("".join(item.text for item in baseline_results))
after_points, after_checks = point_coverage(retrieved_context)

# 上一轮文字不足时，实际再查同一个用户问题；这是“不应直接复用”的对照。
control_topic = next((term for term in ("连续型", "离散型") if term in previous_question), "")
control_is_enough = context_has_topic(retrieved_context, control_topic)
control_counter = {"calls": 0}
control_results = [] if control_is_enough else counted_search(control_counter, previous_question, top_k=2)
control_context = "".join(item.text for item in control_results)

print("追问：", question)
print("上一问实际检索页：", retrieved_pages, "；追问结果页：", baseline_results[0].pages)
print("回答要点覆盖率：", baseline_points, "→", after_points, "；检查：", baseline_checks, "→", after_checks)
print("追问新增资料量：", len(baseline_results), "→", len(after_check_results), "；追问检索次数：", baseline_counter["calls"], "→", after_check_counter["calls"])
print("总检索次数（含上一问）：", previous_counter["calls"] + baseline_counter["calls"], "→", previous_counter["calls"] + after_check_counter["calls"])
print("上下文不足对照：上一问片段含用户主题词？", control_is_enough, "；实际补查次数：", control_counter["calls"], "；补查后片段数：", len(control_results))

assert previous_results and context_is_enough and not after_check_results
assert baseline_points == after_points == 1.0 and control_counter["calls"] == 1 and context_has_topic(control_context, control_topic)
main_annotation = load_annotation(case["id"])
# 改前没有可复用的已核验上下文，改后直接使用上一轮真实检索结果；两边页码仍来自真实结果。
emit_standard('检索前判断是否需要查', 'main', case["id"], [], previous_results, main_annotation["expected_pages"], '再次改善', comparison={'name': '新增检索次数', 'before': baseline_counter['calls'], 'after': after_check_counter['calls'], 'higher_is_better': False})

追问： 根据上面已经核验的段落，离散型标记对应什么任务？
上一问实际检索页： [15] ；追问结果页： [15]
回答要点覆盖率： 1.0 → 1.0 ；检查： [True] → [True]
追问新增资料量： 1 → 0 ；追问检索次数： 1 → 0
总检索次数（含上一问）： 2 → 1
上下文不足对照：上一问片段含用户主题词？ False ；实际补查次数： 1 ；补查后片段数： 2



## 按问题难度限制最多查几轮

“分类里精度和错误率有什么关系”只作为简单对照：一轮已经找全，不把它冒充成难题。真正答不全的是线性判别分析问题，因为它还要求说明广义特征值与特征向量的结论。原问题直接取两个片段时只找到第 41、42 页，仍缺少第 44 页。第二轮根据第一轮确认的方法名补查，才找齐第 41、44 页。

轮次上限先看用户问题的结构：问题中有两个明确的问法，或出现“分别”“对比”“区别”“同时”“进一步”等多要点提示时，最多允许两轮；否则最多一轮。但每一轮结束后还要看实际返回文字是否已经涵盖必要要点，已找全就停止。这里的必要要点只取本题用户问题明确提出的两部分：同类与异类样本的投影关系，以及 N−1 个最大广义特征值与特征向量结论；它是本题的局部检查，不是通用的智能判断器。第二轮具体查什么，仍由第一轮实际出现的方法名和用户问题中未出现的要点决定。

In [4]:
simple_case = cases["accuracy_vs_error_rate"]
complex_case = cases["lda_multihop"]


def question_parts(query):
    # 只按用户问题中的标点和连接语拆分，不读取参考答案或预期页码。
    parts = re.split(r"[？?；;。]|进一步说明|还要说明|并说明", query)
    return [part.strip() for part in parts if part.strip()]


def max_rounds(query):
    markers = ("分别", "请对比", "区别", "同时", "进一步")
    return 2 if len(question_parts(query)) > 1 or any(marker in query for marker in markers) else 1


def retrieve_with_limit(case, follow_up_query=None, fixed_rounds=None, counter=None, required_groups=(), initial_results=None):
    rounds = fixed_rounds if fixed_rounds is not None else max_rounds(case["query"])
    counter = counter if counter is not None else {"calls": 0}
    results = list(initial_results) if initial_results is not None else list(counted_search(counter, case["query"], top_k=1))
    first_text = "".join(item.text for item in results)
    first_round_complete = bool(required_groups) and all(all(term in first_text for term in group) for group in required_groups)
    if rounds == 2 and not first_round_complete:
        results.extend(counted_search(counter, follow_up_query or case["query"], top_k=1))
    return results


simple_first_counter = {"calls": 0}
complex_first_counter = {"calls": 0}
simple_first = retrieve_with_limit(simple_case, fixed_rounds=1, counter=simple_first_counter)
complex_first = retrieve_with_limit(complex_case, fixed_rounds=1, counter=complex_first_counter)
method_name = "线性判别分析" if "线性判别分析" in complex_first[0].text else ""
missing_terms = [
    term for term in ("广义特征值", "特征向量")
    if term in complex_case["query"] and term not in complex_first[0].text
]
complex_follow_up = " ".join([method_name, *missing_terms])
# 这里只依据用户问题明确提出的两部分要求判断是否继续，不做通用的答案质量判断。
requirement_candidates = (("同类样本", "异类样本"), ("广义特征值", "特征向量"))
question_requirements = tuple(
    tuple(term for term in group if term in complex_case["query"])
    for group in requirement_candidates
)
assert question_requirements == requirement_candidates
assert all(term in complex_case["query"] for group in question_requirements for term in group)
complex_one_call_counter = {"calls": 0}
complex_one_call_two_results = counted_search(complex_one_call_counter, complex_case["query"], top_k=2)
simple_routed_counter = {"calls": 0}
complex_routed_counter = {"calls": 0}
simple_routed = retrieve_with_limit(simple_case, counter=simple_routed_counter)
complex_routed = retrieve_with_limit(complex_case, complex_follow_up, counter=complex_routed_counter, required_groups=question_requirements)
simple_always_two_counter = {"calls": 0}
complex_always_two_counter = {"calls": 0}
simple_always_two = retrieve_with_limit(simple_case, fixed_rounds=2, counter=simple_always_two_counter)
complex_always_two = retrieve_with_limit(complex_case, complex_follow_up, fixed_rounds=2, counter=complex_always_two_counter, required_groups=question_requirements)
complex_stop_counter = {"calls": 0}
complex_complete_stop = retrieve_with_limit(complex_case, complex_follow_up, fixed_rounds=2, counter=complex_stop_counter, required_groups=question_requirements, initial_results=complex_routed)

simple_annotation = load_annotation(simple_case["id"])
complex_annotation = load_annotation(complex_case["id"])
always_one = [
    evidence_coverage(simple_first, simple_annotation["expected_pages"]),
    evidence_coverage(complex_first, complex_annotation["expected_pages"]),
]
routed = [
    evidence_coverage(simple_routed, simple_annotation["expected_pages"]),
    evidence_coverage(complex_routed, complex_annotation["expected_pages"]),
]
always_two = [
    evidence_coverage(simple_always_two, simple_annotation["expected_pages"]),
    evidence_coverage(complex_always_two, complex_annotation["expected_pages"]),
]

def complex_answer_coverage(results):
    text = "".join(item.text for item in results)
    projection_goal = all(term in text for term in question_requirements[0])
    eigenvector_result = all(term in text for term in question_requirements[1])
    return (projection_goal + eigenvector_result) / 2

print("简单对照问题：", simple_case["query"])
print("简单问题的问法数量 / 最多轮数：", len(question_parts(simple_case["query"])), "/", max_rounds(simple_case["query"]))
print("简单问题首轮页面覆盖率：", evidence_coverage(simple_first, simple_annotation["expected_pages"]))
print("比较问题的问法数量 / 最多轮数：", len(question_parts(complex_case["query"])), "/", max_rounds(complex_case["query"]))
print("复杂问题原样取两条的页面覆盖率：", evidence_coverage(complex_one_call_two_results, complex_annotation["expected_pages"]))
print("补查问题：", complex_follow_up)
print("复杂问题回答要点覆盖率：", complex_answer_coverage(complex_one_call_two_results), "→", complex_answer_coverage(complex_routed))
print("复杂问题资料量：", len(complex_one_call_two_results), "→", len(complex_routed), "；检索次数：", complex_one_call_counter["calls"], "→", complex_routed_counter["calls"])
print("已有完整资料时的停止检查：回答要点覆盖率", complex_answer_coverage(complex_complete_stop), "；新增检索次数：", complex_stop_counter["calls"])
print("简单对照资料量：", len(simple_first), "；检索次数：", simple_routed_counter["calls"])
print("始终一轮，页面覆盖率：", always_one, "；总检索次数：", simple_first_counter["calls"] + complex_first_counter["calls"], "；总资料量：", len(simple_first) + len(complex_first))
print("按问题分配，页面覆盖率：", routed, "；总检索次数：", simple_routed_counter["calls"] + complex_routed_counter["calls"], "；总资料量：", len(simple_routed) + len(complex_routed))
print("始终两轮，页面覆盖率：", always_two, "；总检索次数：", simple_always_two_counter["calls"] + complex_always_two_counter["calls"], "；总资料量：", len(simple_always_two) + len(complex_always_two))

assert evidence_coverage(simple_first, simple_annotation["expected_pages"]) == 1.0 and max_rounds(simple_case["query"]) == 1
assert always_one == [1.0, 0.5] and routed == [1.0, 1.0] and always_two == [1.0, 1.0]
assert complex_answer_coverage(complex_one_call_two_results) == 0.5 and complex_answer_coverage(complex_routed) == 1.0
assert complex_answer_coverage(complex_complete_stop) == 1.0 and complex_stop_counter["calls"] == 0
difficulty_annotation = complex_annotation
emit_standard('按问题难度限制检索轮次', 'main', complex_case["id"], complex_first, complex_routed, difficulty_annotation["expected_pages"], '确认没有改坏')
emit_standard('按问题难度限制检索轮次', 'check', simple_case["id"], simple_first, simple_routed, simple_annotation["expected_pages"], '确认没有改坏')

简单对照问题： 分类里精度和错误率有什么关系？
简单问题的问法数量 / 最多轮数： 1 / 1
简单问题首轮页面覆盖率： 1.0
比较问题的问法数量 / 最多轮数： 2 / 2
复杂问题原样取两条的页面覆盖率： 0.5
补查问题： 线性判别分析 广义特征值 特征向量
复杂问题回答要点覆盖率： 0.5 → 1.0
复杂问题资料量： 2 → 2 ；检索次数： 1 → 2
已有完整资料时的停止检查：回答要点覆盖率 1.0 ；新增检索次数： 0
简单对照资料量： 1 ；检索次数： 1
始终一轮，页面覆盖率： [1.0, 0.5] ；总检索次数： 2 ；总资料量： 2
按问题分配，页面覆盖率： [1.0, 1.0] ；总检索次数： 3 ；总资料量： 3
始终两轮，页面覆盖率： [1.0, 1.0] ；总检索次数： 4 ；总资料量： 4




## 二次检查：已有特征选择资料时不再检索

第一问已经找到并核对过一段关于特征选择的资料，第二问只是追问其中的含义。再次检索虽然也能返回第 139 页，却是一次重复访问。改动后先检查已有资料是否同时包含“原特征”和“映射变换”，满足就直接回答。这个判断只读取用户问题和已经核对过的资料。

In [5]:
case = cases["feature_selection_reuse_context"]
steps = load_query_controls(case["id"])["steps"]
previous_question = steps[0]["query"]
follow_up = steps[1]["query"]
previous_counter = {"calls": 0}
previous_results = counted_search(previous_counter, previous_question, top_k=1)
retrieved_context = "".join(item.text for item in previous_results)
retrieved_pages = [page for item in previous_results for page in item.pages]
context_is_enough = ("原特征" in retrieved_context or "原来的特征" in retrieved_context) and "映射变换" in retrieved_context
baseline_counter = {"calls": 0}
baseline_results = counted_search(baseline_counter, follow_up, top_k=1)
after_counter = {"calls": 0}
after_results = [] if context_is_enough else counted_search(after_counter, follow_up, top_k=1)

def feature_point_coverage(text):
    compact = "".join(text.split())
    checks = ["特征选择" in compact and ("原特征" in compact or "原来的特征" in compact), "降维" in compact and "映射变换" in compact]
    return sum(checks) / len(checks), checks

before_points, before_checks = feature_point_coverage(baseline_results[0].text)
after_points, after_checks = feature_point_coverage(retrieved_context)
print("追问：", follow_up)
print("上一问实际检索页：", retrieved_pages, "；追问结果页：", baseline_results[0].pages)
print("回答要点覆盖率：", before_points, "→", after_points, "；检查：", before_checks, "→", after_checks)
print("追问新增资料量：", len(baseline_results), "→", len(after_results), "；追问检索次数：", baseline_counter["calls"], "→", after_counter["calls"])
print("总检索次数（含上一问）：", previous_counter["calls"] + baseline_counter["calls"], "→", previous_counter["calls"] + after_counter["calls"])

assert context_is_enough and before_points == after_points == 1.0 and not after_results and after_counter["calls"] == 0
check_annotation = load_annotation(case["id"])
emit_standard('检索前判断是否需要查', 'check', case["id"], baseline_results, previous_results, check_annotation["expected_pages"], '再次改善', comparison={'name': '新增检索次数', 'before': baseline_counter['calls'], 'after': after_counter['calls'], 'higher_is_better': False})

追问： 根据刚才核验的段落，降维后的特征还是原来的特征吗？
上一问实际检索页： [139] ；追问结果页： [139]
回答要点覆盖率： 1.0 → 1.0 ；检查： [True, True] → [True, True]
追问新增资料量： 1 → 0 ；追问检索次数： 1 → 0
总检索次数（含上一问）： 2 → 1



## 结论

第一项把已有资料后的重复检索从一次降为零。第二项在两道题都找全页面的情况下，比“所有问题都查两轮”少一次检索，也避免了“所有问题都只查一轮”漏掉复杂问题。

第二轮特征选择追问确认：第 139 页已经核验过的上下文足以回答，因此追问检索从一次降为零，且两项回答要点保持完整。这里的轮次规则只是根据问题的分问结构给出上限；首轮概念补查则是首轮已经返回内容后，针对其中出现的概念再补查。两者都还要在每轮后检查回答要点，不能把“允许再查”当成“必然要再查”。示例规则只处理这里的几类问题，不能直接照搬；实际使用时仍应根据自己的问题调整判断词。